In [2]:
%pip install --upgrade langgraph openai python-dotenv

  Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl.metadata (14 kB)
  Using cached zstandard-0.25.0-cp312-cp312-win_amd64.whl.metadata (3.3 kB)
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ------------------------------ --------- 1.8/2.4 MB 11.2 MB/s eta 0:00:01
   ---------------------------------------- 2.4/2.4 MB 11.6 MB/s eta 0:00:00
   ---------------------------------------- 0.0/580.5 kB ? eta -:--:--
   ---------------------------------------- 580.5/580.5 kB 4.1 MB/s eta 0:00:00
   ---------------------------------------- 0.0/940.8 kB ? eta -:--:--
   ---------------------------------------- 940.8/940.8 kB 8.7 MB/s eta 0:00:00
Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl (54 kB)
Using cached zstandard-0.25.0-cp312-cp312-win_amd64.whl (506 kB)
  Attempting uninstall: jsonpointer
    Found existing installation: jsonpointer 3.1.1
    Uninstalling jsonpointer-3.1.1:
      Successfully uninstalled jsonpointer-3.1.1
  Attempting uninst

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
crewai 1.15.30 requires openai<3,>=2.30.0, but you have openai 3.46.0 which is incompatible.
instructor 1.15.4 requires jiter<0.15,>=0.6.1, but you have jiter 0.18.0 which is incompatible.
instructor 1.15.4 requires openai<3.0.0,>=2.0.0, but you have openai 3.46.0 which is incompatible.

[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
#Importing libraries
import os
from typing import TypedDict

from dotenv import load_dotenv
from openai import OpenAI

from langgraph.graph import StateGraph, START, END

load_dotenv()

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

In [4]:
#Defining Workflow state

class ResearchState(TypedDict):
    goal: str
    search_results: list[dict]
    key_points: list[str]
    summary: str

In [5]:
#First Node : 
knowledge_base = [
    {
        "topic": "RAG",
        "text": "Retrieval-Augmented Generation combines information retrieval with language generation. It retrieves relevant documents and provides them to an LLM as context."
    },
    {
        "topic": "Embeddings",
        "text": "Embeddings convert text into numerical vectors. Similar meanings can have vectors that are close together in vector space."
    },
    {
        "topic": "Vector databases",
        "text": "Vector databases store embeddings and support similarity searches to retrieve relevant information."
    },
    {
        "topic": "Chunking",
        "text": "Chunking splits large documents into smaller pieces. This helps retrieval systems find relevant information more precisely."
    }
]

In [6]:
#Creating Search Node
def search_node(state: ResearchState):
    query_words = set(state["goal"].lower().split())

    results = []

    for document in knowledge_base:
        document_text = document["text"].lower()

        score = sum(
            word in document_text
            for word in query_words
        )

        if score > 0:
            results.append({
                "topic": document["topic"],
                "text": document["text"],
                "score": score
            })

    results.sort(
        key=lambda item: item["score"],
        reverse=True
    )

    return {
        "search_results": results[:3]
    }

In [7]:
#Second Node : extract key points
def extract_node(state: ResearchState):
    points = []

    for result in state["search_results"]:
        point = (
            result["topic"] + ": " + result["text"]
        )

        points.append(point)

    return {
        "key_points": points
    }

In [8]:
#Third Node: Generate a Summary
def summary_node(state: ResearchState):

    context = "\n".join(
        state["key_points"]
    )

    response = client.responses.create(
        model="gpt-4.1-mini",
        instructions=(
            "Write a concise beginner-friendly summary. "
            "Use only the supplied information. "
            "Do not invent unsupported facts."
        ),
        input=f"""
        Research question:
        {state["goal"]}

        Retrieved information:
        {context}
        """
    )

    return {
        "summary": response.output_text
    }

In [10]:
#Adding a Conditional Branch. A useful workflow should not try to summarize when no documents were found.
#Let's create a node for that situation:
def no_results_node(state: ResearchState):
    return {
        "summary": (
            "No matching documents were found. "
            "Try a different research question."
        )
    }


In [11]:
#Routing function
def route_after_search(state: ResearchState):

    if state["search_results"]:
        return "extract"

    return "no_results"

In [12]:
#Building the langgraph
workflow = StateGraph(ResearchState)

# Register nodes
workflow.add_node("search", search_node)
workflow.add_node("extract", extract_node)
workflow.add_node("summarize", summary_node)
workflow.add_node("no_results", no_results_node)

# Define the starting point
workflow.add_edge(START, "search")

# Add conditional routing
workflow.add_conditional_edges(
    "search",
    route_after_search,
    {
        "extract": "extract",
        "no_results": "no_results"
    }
)

# Connect the remaining steps
workflow.add_edge("extract", "summarize")

# Define the ending points
workflow.add_edge("summarize", END)
workflow.add_edge("no_results", END)

# Compile the graph
app = workflow.compile()

In [13]:
#Running the agent
initial_state = {
    "goal": "Explain RAG, embeddings, and vector databases",
    "search_results": [],
    "key_points": [],
    "summary": ""
}

result = app.invoke(initial_state)

In [14]:
print("Research question:")
print(result["goal"])

print("\nRetrieved documents:")
for item in result["search_results"]:
    print("-", item["topic"])

print("\nKey points:")
for point in result["key_points"]:
    print("-", point)

print("\nFinal summary:")
print(result["summary"])

Research question:
Explain RAG, embeddings, and vector databases

Retrieved documents:
- Vector databases
- RAG
- Embeddings

Key points:
- Vector databases: Vector databases store embeddings and support similarity searches to retrieve relevant information.
- RAG: Retrieval-Augmented Generation combines information retrieval with language generation. It retrieves relevant documents and provides them to an LLM as context.
- Embeddings: Embeddings convert text into numerical vectors. Similar meanings can have vectors that are close together in vector space.

Final summary:
RAG (Retrieval-Augmented Generation) is a method that improves language models by first retrieving relevant documents and then using them as context to generate better responses. Embeddings are numerical representations of text, where similar meanings are close together as vectors. Vector databases store these embeddings and allow similarity searches to quickly find relevant information. Together, these tools help comb

In [15]:
#Trying a question that doesn't exist in out knowledge base
no_result = app.invoke({
    "goal": "Explain quantum computing",
    "search_results": [],
    "key_points": [],
    "summary": ""
})

print(no_result["summary"])

No matching documents were found. Try a different research question.
